In [2]:
# Setup

%load_ext autoreload
%autoreload 2

import numpy as np
import onnxruntime
from skl2onnx.common.data_types import FloatTensorType
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from pipelines.god_di.com_dataset.com_parser import ComDatasetParser
from pipelines.god_di.com_dataset.labeling import (
    ComDatasetLabeler,
    DirectLabelValueMapping,
    ComparisonDirection,
    undersample,
)
from skl2onnx import convert_sklearn
import pandas as pd
from pandas import Series

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Read CSV dataset

annotations = [
    "@Inject",
    "@Autowired",
    "@Service",
    "@Repository",
    "@Controller",
    "@RestController",
    "@Component",
]
input_file = input("Enter path to the file: ")


def check_annotation(idx: int, df: Series) -> bool:
    code = df["Code"]

    if any(annotation in code for annotation in annotations):
        print(f"{idx}: {code}")


data_frame = pd.read_csv(filepath_or_buffer=input_file)
for index, row in data_frame.iterrows():
    check_annotation(index, row)

In [ ]:
# input_file = input("Enter path to the file: ")
# input_file = "/media/martin/BigData/datasets/code_smell_quality_attribute/24057336/CodeSmellsDataset/Class/Blob_CombinedAll_MLCQ_Maiga_Labeled_CleanedByMedian_Shuffled.csv"
# input_file = "/media/martin/BigData/datasets/code_smell_quality_attribute/24057336/CodeSmellsDataset/Class/LargeClass_CombinedAll_Labeled_CleanedByMedian_Shuffled.csv"
# input_file = "/media/martin/BigData/datasets/code_smell_quality_attribute/24057336/CodeSmellsDataset/Class/ComplexClass_CombinedAll_Labeled_CleanedByMedian_Shuffled.csv"
# input_file = "/media/martin/BigData/datasets/pa165-git/all-metrics-no-code.jsonl"
input_file = "/media/martin/BigData/datasets/smelly-code-++/28519385/52714583_multi-smell-dataset-v1_2.csv"
# data_frame = pd.read_json(input_file, lines=True)
data_frame = pd.read_csv(filepath_or_buffer=input_file)
print(data_frame.loc[0])
# data_frame.query("injectedFields >= 5").query("methodsCount > 5")

In [ ]:
# CAM dataset parsing - https://github.com/yegor256/cam?tab=readme-ov-file

root_file = "/media/martin/BigData/datasets/cam_2025/data"
output_file = "/media/martin/BigData/datasets/cam_2025/parsed.jsonl"

# root_file = input("Enter root file of CAM dataset: ")
# output_file = input("Enter output file (.jsonl): ")
include_files = ["cc", "LCOM5"]

parser = ComDatasetParser(output_path=output_file)
parser.parse_com_dateset(root_file, ["cc", "LCOM5", "nooa", "noom", "nocm", "loc"])

In [ ]:
# com_dataset_path = input("Enter com file path (.jsonl): ")
com_dataset_path = "/media/martin/BigData/datasets/cam_2025/parsed.jsonl"

data_frame = pd.read_json(com_dataset_path, lines=True)
# data_frame["nocm"] = data_frame["nocm"].fillna(0.0)
# data_frame = data_frame[~data_frame["java_file"].str.contains("generated-sources")]
# data_frame.query("global_LCOM5_pct > 0.8").sort_values(by="LCOM5", ascending=True)
data_frame.query("nooa <= 4").sort_values(by="nooa", ascending=True)

In [ ]:
# CoM dataset labeling

metric_percentile_mapping: dict[str, float] = {
    "loc": 0.9,
    "cc": 0.95,
}

value_label_mapping: list[DirectLabelValueMapping] = [
    DirectLabelValueMapping(
        metric="noom", value=50, direction=ComparisonDirection.greater_than, label=1
    ),
    DirectLabelValueMapping(
        metric="nooa", value=15, direction=ComparisonDirection.greater_than, label=1
    ),
    DirectLabelValueMapping(
        metric="nocm", value=15, direction=ComparisonDirection.greater_than, label=1
    ),
    DirectLabelValueMapping(
        metric="nooa", value=15, direction=ComparisonDirection.greater_than, label=1
    ),
    DirectLabelValueMapping(
        metric="nooa", value=4, direction=ComparisonDirection.lower_than, label=0
    ),
    DirectLabelValueMapping(
        metric="LCOM5", value=0.2, direction=ComparisonDirection.lower_than, label=0
    ),
]
# Greater index = higher priority

labeler = ComDatasetLabeler(metric_percentile_mapping, value_label_mapping)

com_dataset_path = "/media/martin/BigData/datasets/cam_2025/parsed.jsonl"
output_file = "/media/martin/BigData/datasets/cam_2025/parsed-labeled.jsonl"

data_frame = pd.read_json(com_dataset_path, lines=True)

labeler.do_labeling(data_frame)

# Undersample 0
data_frame = undersample(data_frame, frac=0.5, label="label", value=0)
data_frame.to_json(output_file, lines=True, orient="records")
# data_frame.loc[data_frame["label"] == 0]

In [3]:
# Training the model

dataset_file = "/media/martin/BigData/datasets/cam_2025/parsed-labeled.jsonl"

data_frame = pd.read_json(dataset_file, lines=True)

features = ["noom", "nooa", "nocm", "LCOM5", "cc", "loc"]
x_values = data_frame[features]
labels = data_frame["label"]

x_train, x_test, y_train, y_test = train_test_split(
    x_values, labels, test_size=0.2, random_state=42
)

classifier = RandomForestClassifier(random_state=42)
pipeline = Pipeline(steps=[("classifier", classifier)])

pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

print(y_pred[:5])

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:2f} %")

f1 = f1_score(y_test, y_pred)
print(f"F1 score: {f1 * 100:2f} %")

[0 0 0 0 1]
Accuracy: 99.835120 %
F1 score: 99.415205 %


In [4]:
# Export to ONNX format (dependent on pipeline from previous cell)

onnx_path = input("Enter onnx output path: ")
if len(onnx_path) == 0:
    exit(0)

input_types = [("inputs", FloatTensorType([None, len(features)]))]
print(input_types)

onnx_model = convert_sklearn(
    pipeline, "god_di_classifier", input_types, target_opset=12
)

with open(onnx_path, "wb") as file:
    file.write(onnx_model.SerializeToString())

[('inputs', FloatTensorType(shape=[None, 6]))]


In [22]:
# ONNX predictions

# features = ["noom", "nooa", "nocm", "LCOM5", "cc", "loc"]
# inputs = x_test[features].to_numpy(dtype=np.float32)
inputs = np.array([[25, 9, 0, 1.0, 4, 250]], dtype=np.float32)

print(inputs)
sess = onnxruntime.InferenceSession(
    "/media/martin/BigData/models/god_di/god_di_onnx/model.onnx",
    providers=["CPUExecutionProvider"],
)
pred_onx = sess.run(None, {"inputs": inputs})
print("predict", pred_onx[0])
# print("predict_proba", pred_onx[1][:2])

[[ 25.   9.   0.   1.   4. 250.]]
predict [1]
